# 🔬 Model Pipeline: Qwen-2.5-7B-Instruct (4-bit QLoRA)
### **Jadavpur University Research Internship | Under: Prof. Tohida Rehman**
**Track:** FIRE MixSub-SciHigh Scientific Highlight Summarization
**Model ID:** `Qwen/Qwen2.5-7B-Instruct`
**Architecture:** CAUSAL (4-bit QLoRA)

---
### 🎯 Strict Uniform Benchmark Constraints (Scientifically Fair)
- **`MAX_SOURCE_LENGTH = 512` tokens**: Uniform input length across all 6 models.
- **`MAX_TARGET_LENGTH = 128` tokens**: Uniform output generation limit across all 6 models.

### 📊 The 3 Non-Redundant Benchmark Pillars Measured:
1. **Overall Quality**: ROUGE-L F1 (↑ Higher is Better) & Target F1 Score (↑)
2. **Hallucination Risk**: Ungrounded Token % (↓ Lower is Better) & Precision Source (% grounded in abstract)
3. **Inference Time**: Wall-Clock Latency per Sample (↓ Lower is Faster) & Throughput (tok/s ↑)

In [ ]:

# CELL 1: Google Drive setup

from google.colab import drive
import os

drive.mount("/content/drive")

DRIVE_BASE_DIR = "/content/drive/MyDrive/JU_Benchmark_Project"
DATASET_DIR = os.path.join(DRIVE_BASE_DIR, "Dataset")
RESULTS_DIR = os.path.join(DRIVE_BASE_DIR, "results")

os.makedirs(DATASET_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

MAX_SOURCE_LENGTH = 512
MAX_TARGET_LENGTH = 128

TRAIN_CSV = os.path.join(
    DATASET_DIR, "MixSub-SciHigh_train_FIRE.csv"
)
VAL_CSV = os.path.join(
    DATASET_DIR, "MixSub-SciHigh_val_FIRE.csv"
)
TEST_CSV = os.path.join(
    DATASET_DIR, "MixSub-SciHigh_test_FIRE(Masked).csv"
)

print("Drive ready.")
print("Model:", MODEL_ID)
print("Results:", RESULTS_DIR)


In [ ]:

# CELL 2: Install dependencies and check GPU

%pip install -q transformers accelerate bitsandbytes rouge-score

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU not detected. Select Runtime > Change runtime type > T4 GPU."
    )

print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)

COMPUTE_DTYPE = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)

print("Compute dtype:", COMPUTE_DTYPE)


In [ ]:

# CELL 3: Dataset loading and three-pillar metric engine

import pandas as pd
import numpy as np
import re
from collections import Counter
from rouge_score import rouge_scorer

for path in [VAL_CSV]:
    if not os.path.isfile(path):
        raise FileNotFoundError(f"Required validation file missing: {path}")

def clean_dataset(df, require_highlights=True):
    required = ["Abstract", "Highlights"] if require_highlights else ["Abstract"]

    missing = set(required) - set(df.columns)
    if missing:
        raise ValueError(f"Missing dataset columns: {missing}")

    df = df.dropna(subset=required).copy()

    df["Abstract"] = df["Abstract"].astype(str).str.strip()

    if require_highlights:
        df["Highlights"] = df["Highlights"].astype(str).str.strip()
        df = df[
            (df["Abstract"] != "") &
            (df["Highlights"] != "")
        ]
    else:
        df = df[df["Abstract"] != ""]

    return df.reset_index(drop=True)

val_df = clean_dataset(pd.read_csv(VAL_CSV))

test_df = None
if os.path.isfile(TEST_CSV):
    test_df = clean_dataset(
        pd.read_csv(TEST_CSV),
        require_highlights=False
    )
    print("Masked test samples:", len(test_df))
else:
    print("Masked test CSV not found. Test generation will be skipped.")

print("Validation samples:", len(val_df))

# Metric functions
def clean_tokenize(text):
    return re.findall(
        r"\b[a-zA-Z0-9]+(?:'[a-zA-Z0-9]+)?\b",
        str(text).lower()
    )

def compute_pillars_for_sample(pred, source, reference):
    pred_counts = Counter(clean_tokenize(pred))
    source_counts = Counter(clean_tokenize(source))
    ref_counts = Counter(clean_tokenize(reference))

    total_pred = sum(pred_counts.values())
    total_ref = sum(ref_counts.values())

    source_overlap = sum((pred_counts & source_counts).values())
    ref_overlap = sum((pred_counts & ref_counts).values())

    source_precision = (
        source_overlap / total_pred if total_pred else 0.0
    )
    target_precision = (
        ref_overlap / total_pred if total_pred else 0.0
    )
    target_recall = (
        ref_overlap / total_ref if total_ref else 0.0
    )

    target_f1 = (
        2 * target_precision * target_recall /
        (target_precision + target_recall)
        if target_precision + target_recall else 0.0
    )

    return {
        "source_grounding_precision": source_precision,
        "hallucination_risk": 1.0 - source_precision,
        "precision_target": target_precision,
        "recall_target": target_recall,
        "f1_target": target_f1
    }

def evaluate_core_benchmarks(
    predictions, sources, references, latencies, token_counts
):
    if not predictions:
        raise ValueError("No predictions to evaluate.")

    scorer = rouge_scorer.RougeScorer(
        ["rougeL"], use_stemmer=True
    )

    rouge_scores = [
        scorer.score(ref, pred)["rougeL"]
        for pred, ref in zip(predictions, references)
    ]

    pillar_scores = [
        compute_pillars_for_sample(pred, src, ref)
        for pred, src, ref in zip(predictions, sources, references)
    ]

    total_time = sum(latencies)
    total_tokens = sum(token_counts)

    return {
        "overall_quality_rougeL_f1": float(
            np.mean([x.fmeasure for x in rouge_scores])
        ),
        "rougeL_precision": float(
            np.mean([x.precision for x in rouge_scores])
        ),
        "rougeL_recall": float(
            np.mean([x.recall for x in rouge_scores])
        ),
        "f1_score_target": float(
            np.mean([x["f1_target"] for x in pillar_scores])
        ),
        "precision_target": float(
            np.mean([x["precision_target"] for x in pillar_scores])
        ),
        "recall_target": float(
            np.mean([x["recall_target"] for x in pillar_scores])
        ),
        "source_grounding_precision": float(
            np.mean([x["source_grounding_precision"] for x in pillar_scores])
        ),
        "hallucination_risk_pct": float(
            np.mean([x["hallucination_risk"] for x in pillar_scores])
        ),
        "wall_clock_latency_sec": float(
            np.mean(latencies)
        ),
        "total_wall_clock_sec": float(total_time),
        "throughput_tokens_per_sec": float(
            total_tokens / max(total_time, 1e-8)
        )
    }

print("Dataset and metric engine ready.")


In [ ]:

# FAST CELL 4: Qwen2.5-1.5B-Instruct

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)

model.eval()

print("Qwen2.5-1.5B loaded successfully.")


In [ ]:

# CELL 5: Qwen zero-shot evaluation and Drive export

import time
import json
import gc

MAX_EVAL_SAMPLES = 20

eval_df = val_df.copy()

if MAX_EVAL_SAMPLES is not None:
    eval_df = eval_df.head(MAX_EVAL_SAMPLES).copy()

def build_prompt(abstract):
    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert scientific researcher. "
                "Generate concise, accurate scientific research "
                "highlights using only the information in the abstract. "
                "Do not invent facts."
            )
        },
        {
            "role": "user",
            "content": "Abstract:\n" + str(abstract).strip()
        }
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

def generate_highlight(abstract):
    prompt = build_prompt(abstract)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SOURCE_LENGTH
    )

    input_device = model.get_input_embeddings().weight.device
    inputs = {
        key: value.to(input_device)
        for key, value in inputs.items()
    }

    prompt_length = inputs["input_ids"].shape[1]

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    start = time.perf_counter()

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_TARGET_LENGTH,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True
        )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - start

    generated_ids = output[0][prompt_length:]

    prediction = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    return prediction, elapsed, len(generated_ids)


# Warm-up
print("Warming up Qwen...")
_ = generate_highlight(
    "A short scientific abstract for model warm-up."
)

# Validation inference
sources = eval_df["Abstract"].tolist()
references = eval_df["Highlights"].tolist()

predictions = []
latencies = []
token_counts = []

print(f"\nEvaluating {len(eval_df)} validation samples...")

for i, abstract in enumerate(sources):
    pred, latency, token_count = generate_highlight(abstract)

    predictions.append(pred)
    latencies.append(latency)
    token_counts.append(token_count)

    print(
        f"Sample {i+1}/{len(sources)} | "
        f"Latency: {latency:.2f}s"
    )

# Calculate benchmark metrics
metrics = evaluate_core_benchmarks(
    predictions,
    sources,
    references,
    latencies,
    token_counts
)

metrics.update({
    "model": "qwen-2.5-7b",
    "model_id": MODEL_ID,
    "max_source_length": MAX_SOURCE_LENGTH,
    "max_target_length": MAX_TARGET_LENGTH,
    "evaluation_samples": len(predictions),
    "fine_tuning": False,
    "quantization": "4-bit NF4",
    "evaluation_type": "Zero-shot inference"
})

# Save benchmark JSON
benchmark_path = os.path.join(
    RESULTS_DIR,
    "benchmark_qwen-2.5-7b.json"
)

with open(benchmark_path, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=4, ensure_ascii=False)

# Save validation predictions
validation_output = eval_df.copy()
validation_output["Generated_Highlights"] = predictions
validation_output["Inference_Latency_Sec"] = latencies
validation_output["Generated_Tokens"] = token_counts

validation_path = os.path.join(
    RESULTS_DIR,
    "qwen_validation_predictions.csv"
)

validation_output.to_csv(validation_path, index=False)

print("\nValidation benchmark saved.")
print(json.dumps(metrics, indent=4))

# Optional masked test prediction
if test_df is not None:
    print(f"\nGenerating {len(test_df)} masked test predictions...")

    test_predictions = []

    for i, abstract in enumerate(test_df["Abstract"].tolist()):
        pred, _, _ = generate_highlight(abstract)
        test_predictions.append(pred)

        if (i + 1) % 25 == 0 or i + 1 == len(test_df):
            print(f"Test progress: {i+1}/{len(test_df)}")

    test_output = test_df.copy()
    test_output["Highlights"] = test_predictions

    test_path = os.path.join(
        RESULTS_DIR,
        "MixSub-SciHigh_test_predictions_qwen-2.5-7b.csv"
    )

    test_output.to_csv(test_path, index=False)
    print("Test predictions saved:", test_path)

# Verify Drive files
print("\n--- DRIVE OUTPUT VERIFICATION ---")

for path in [benchmark_path, validation_path]:
    print(
        os.path.basename(path),
        "| Exists:", os.path.isfile(path),
        "| Size:", os.path.getsize(path), "bytes"
    )

if test_df is not None:
    print(
        os.path.basename(test_path),
        "| Exists:", os.path.isfile(test_path)
    )

print("\nQwen-2.5-7B zero-shot pipeline completed.")

# Free GPU memory when finished
del model
gc.collect()
torch.cuda.empty_cache()
